In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# 加载数据
order_data = pd.read_csv('order_detail.csv')
visit_data = pd.read_csv('user_visit_detail.csv')
coupon_data = pd.read_csv('user_coupon_receive.csv')

In [ ]:
visit_data.head()

In [ ]:
coupon_data.head()

In [ ]:
order_data.head()

In [ ]:
# 数据预处理
# 确保日期字段是日期格式
order_data['Pay_date'] = pd.to_datetime(order_data['Pay_date'])
visit_data['Visit_date'] = pd.to_datetime(visit_data['Visit_date'])
coupon_data['Receive_date'] = pd.to_datetime(coupon_data['Receive_date'])
coupon_data['Start_date'] = pd.to_datetime(coupon_data['Start_date'])
coupon_data['End_date'] = pd.to_datetime(coupon_data['End_date'])

In [ ]:
# 生成用户活跃天数特征
visit_counts = visit_data.groupby('User_id')['Visit_date'].nunique().reset_index()
visit_counts.columns = ['User_id', 'Visit_days']

In [ ]:
# 生成用户的订单数量的特征
user_order_counts = order_data.groupby('User_id')['Order_id'].count().reset_index()
user_order_counts.columns = ['User_id', 'Order_count']

In [ ]:
# 生成用户获取的优惠券数量特征
coupon_counts = coupon_data.groupby('User_id')['Coupon_id'].count().reset_index()
coupon_counts.columns = ['User_id', 'Coupon_count']

In [ ]:
# 生成用户使用优惠券的数量特征
coupon_usage = coupon_data[coupon_data['Coupon_status'] == 2].groupby('User_id')['Coupon_id'].count().reset_index()
coupon_usage.columns = ['User_id', 'Coupon_usage_count']

In [ ]:
# 合并订单数据和用户活跃天数特征
merged_data = pd.merge(order_data, visit_counts, on='User_id', how='left')
merged_data.head()

In [ ]:
# 合并订单数据和订单数量特征
merged_data = pd.merge(merged_data, user_order_counts, on='User_id', how='left')
merged_data.head()

In [ ]:
# 合并订单数据和用户获取的优惠券数量特征
merged_data = pd.merge(merged_data, coupon_counts, on='User_id', how='left')
merged_data.head()

In [ ]:
# 合并订单数据和用户使用优惠券的数量特征
merged_data = pd.merge(merged_data, coupon_usage, on='User_id', how='left')
merged_data.head()

In [ ]:
# 合并订单数据和用户获得优惠券特征
merged_data = pd.merge(merged_data, coupon_data, on='Coupon_id', how='left')
merged_data.head()

In [ ]:
merged_data.describe()

In [ ]:
# 填充缺失值
merged_data['Visit_days'].fillna(0, inplace=True)
merged_data['Coupon_count'].fillna(0, inplace=True)
merged_data['Coupon_usage_count'].fillna(0, inplace=True)

In [ ]:
# 保存合并后的数据
merged_data.to_csv('merged_data.csv', index=False)

In [ ]:
##异常值处理
merged_data = pd.read_csv('merged_data.csv')
print(merged_data.isnull().sum())

In [ ]:
# 缺失值
# 删除user_id_y
merged_data = merged_data.drop(columns=['User_id_y'])

In [ ]:
# 将user_id_x名称改为user_id
merged_data = merged_data.rename(columns={'User_id_x': 'User_id'})

In [ ]:
# 将visit_days和Coupon_count和Coupon_usage_count的缺失值填充为0
merged_data['Visit_days'] = merged_data['Visit_days'].fillna(0)
merged_data['Coupon_count'] = merged_data['Coupon_count'].fillna(0)
merged_data['Coupon_usage_count'] = merged_data['Coupon_usage_count'].fillna(0)

In [ ]:
# 标记Coupon_id、Coupon_type、Coupon_status、Coupon_amt、Receive_date、Start_date、End_date和Price_limit不同时为空或非空的值为异常值
columns_to_check = ['Coupon_id', 'Coupon_type', 'Coupon_status', 'Coupon_amt', 'Receive_date', 'Start_date', 'End_date', 'Price_limit']
merged_data['is_anomaly'] = merged_data.apply(lambda row: all(pd.isna(row[columns_to_check]) == pd.isna(row[columns_to_check[0]])), axis=1)
merged_data['is_anomaly'] = merged_data['is_anomaly'].apply(lambda x: 0 if x else 1)

In [ ]:
# 保存处理后的数据
merged_data.to_csv('processed_data.csv', index=False)

In [ ]:

# 加载处理后的数据
processed_data = pd.read_csv('processed_data.csv')
print(processed_data.info())
print(processed_data.describe())
print(processed_data.isnull().sum())

# 筛选出is_anomaly=1的行
anomalies = processed_data[processed_data['is_anomaly'] == 1]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 查看数据的基本信息
print(merged_data.info())
print(merged_data.describe())

# 查看数据的前几行
print(merged_data.head())

In [ ]:
# 绘制用户活跃天数的分布
plt.figure(figsize=(10, 6))
sns.histplot(merged_data['Visit_days'], bins=30, kde=True)
plt.title('Distribution of User Visit Days')
plt.xlabel('Visit Days')
plt.ylabel('Frequency')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde

# 绘制用户活跃天数的分布
plt.figure(figsize=(10, 6))

# 绘制直方图
plt.hist(merged_data['Visit_days'], bins=30, density=True, alpha=0.6, color='blue', edgecolor='black')

# 绘制核密度估计（KDE）曲线
kde = gaussian_kde(merged_data['Visit_days'].dropna())  # 计算核密度估计
x_range = np.linspace(merged_data['Visit_days'].min(), merged_data['Visit_days'].max(), 1000)  # 生成x轴的范围
plt.plot(x_range, kde(x_range), color='red', linewidth=2)  # 绘制KDE曲线

# 添加标题和标签
plt.title('Distribution of User Visit Days')
plt.xlabel('Visit Days')
plt.ylabel('Density')

# 显示图形
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde

# 绘制用户获取优惠券数量的分布
plt.figure(figsize=(10, 6))

# 绘制直方图
hist, bins, _ = plt.hist(merged_data['Coupon_count'], bins=30, density=True, alpha=0.6, color='blue', edgecolor='black')

# 绘制核密度估计（KDE）曲线
kde = gaussian_kde(merged_data['Coupon_count'].dropna())  # 计算核密度估计
x_range = np.linspace(merged_data['Coupon_count'].min(), merged_data['Coupon_count'].max(), 1000)  # 生成x轴的范围
plt.plot(x_range, kde(x_range), color='red', linewidth=2)  # 绘制KDE曲线

# 添加标题和标签
plt.title('Distribution of Coupon Counts')
plt.xlabel('Coupon Count')
plt.ylabel('Density')

# 显示图形
plt.show()

In [ ]:
merged_data['Coupon_count'].mode()

In [ ]:
# 绘制用户使用优惠券数量的分布
plt.figure(figsize=(10, 6))
sns.histplot(merged_data['Coupon_usage_count'], bins=30, kde=True)
plt.title('Distribution of Coupon Usage Counts')
plt.xlabel('Coupon Usage Count')
plt.ylabel('Frequency')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde

# 绘制用户获取优惠券数量的分布
plt.figure(figsize=(10, 6))

# 绘制直方图
hist, bins, _ = plt.hist(merged_data['Coupon_usage_count'], bins=30, density=True, alpha=0.6, color='blue', edgecolor='black')

# 绘制核密度估计（KDE）曲线
kde = gaussian_kde(merged_data['Coupon_usage_count'].dropna())  # 计算核密度估计
x_range = np.linspace(merged_data['Coupon_usage_count'].min(), merged_data['Coupon_usage_count'].max(), 1000)  # 生成x轴的范围
plt.plot(x_range, kde(x_range), color='red', linewidth=2)  # 绘制KDE曲线

# 添加标题和标签
plt.title('Distribution of Coupon_usage_count')
plt.xlabel('Coupon_usage_count')
plt.ylabel('Density')

# 显示图形
plt.show()

In [ ]:
##特征提取
merged_data = pd.read_csv('processed_data.csv')
print(merged_data.columns)

# 转换日期字段
merged_data['Pay_date'] = pd.to_datetime(merged_data['Pay_date'])
merged_data['Receive_date'] = pd.to_datetime(merged_data['Receive_date'])
#按照用户——支付时间排序
merged_data = merged_data.sort_values(by=['User_id', 'Pay_date'])

In [ ]:
merged_data.head()

In [ ]:
# 特征生成
# 生成用户平均客单价特征
merged_data['Avg_order_amount'] = merged_data['Actual_pay'] / merged_data['Order_count']
merged_data['Avg_order_amount'].fillna(0, inplace=True)

In [ ]:
# 1. 使用的优惠券占总优惠券的比例
merged_data['Coupon_usage_rate'] = merged_data['Coupon_usage_count'] / merged_data['Coupon_count']
merged_data['Coupon_usage_rate'].fillna(0, inplace=True)

In [ ]:
# 2. 高频使用优惠券
merged_data['coupon_usage_interval'] = merged_data.groupby('User_id')['Pay_date'].diff().dt.days.fillna(0)

In [ ]:
# 3. 低客单价高补贴
# 计算补贴金额占实际支付金额的比例
merged_data['subsidy_ratio'] = merged_data['Reduce_amount'] / merged_data['Actual_pay']
# 标记低客单价高补贴
# 假设补贴金额占实际支付金额的比例超过50%为高补贴
merged_data['low_price_high_subsidy'] = np.where(merged_data['subsidy_ratio'] > 0.5, 1, 0)

In [ ]:
# 4. 频繁登录但订单少
merged_data['login_order_ratio'] = merged_data.groupby('User_id')['Visit_days'].transform(lambda x: x / x.max()) / merged_data.groupby('User_id')['Order_count'].transform(lambda x: x / x.max())

In [ ]:
# 5. 大量获取未使用优惠券
merged_data['unused_coupon_count'] = merged_data['Coupon_count'] - merged_data['Coupon_usage_count']

In [ ]:
# 6. 短时间内大量获取优惠券
merged_data['coupon_acquisition_rate'] = merged_data.groupby('User_id')['Receive_date'].transform(lambda x: x.diff().dt.days.fillna(0).apply(lambda y: 1 if y < 7 else 0))

In [ ]:
#7、负利润订单
merged_data['rule2_free_order'] = np.where(
        (merged_data['Actual_pay'] <= 0) | 
        (merged_data['subsidy_ratio'] >= 1), 1, 0)

In [ ]:
merged_data['rule2_free_order'].value_counts()

In [ ]:
# 计算用户最后活跃日期和平均活跃天数
user_last_active = merged_data.groupby('User_id')['Pay_date'].max().reset_index(name='last_active')
user_historical_activity = merged_data.groupby('User_id')['Visit_days'].mean().reset_index(name='avg_visit_days')

# 合并到原始数据
merged_data = merged_data.merge(user_last_active, on='User_id')
merged_data = merged_data.merge(user_historical_activity, on='User_id')

# 标记历史低活跃但近期（7天内）下单的用户
latest_date = merged_data['Pay_date'].max()
merged_data['rule3_dormant'] = np.where(
    (merged_data['avg_visit_days'] <= 3) & 
    ((latest_date - merged_data['last_active']).dt.days <= 7), 1, 0)

# 删除临时列
merged_data.drop(['last_active', 'avg_visit_days'], axis=1, inplace=True)

In [ ]:
merged_data['rule3_dormant'].value_counts()

In [ ]:
# 标记领券后5分钟内使用的订单（假设时间单位为小时）
merged_data['rule4_rapid_usage'] = np.where(
    merged_data['coupon_usage_interval'] < 24, 1, 0)  # 0.083小时≈5分钟

In [ ]:
merged_data['rule4_rapid_usage'].value_counts()

In [ ]:
#解决中文显示问题
plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde

# 过滤有效数据（排除未使用券和异常值）
valid_data = merged_data[
    (merged_data['coupon_usage_interval'] > 0) &  # 有效使用间隔
    (merged_data['coupon_usage_interval'] != np.inf)  # 排除无穷大
]['coupon_usage_interval']

# 创建画布
plt.figure(figsize=(14, 7))

# 动态分箱策略（小时单位）
max_hours = int(valid_data.max())  # 最大小时数
bin_step = 6 if max_hours > 72 else 1  # 超过3天按6小时分箱，否则按1小时
bins = np.arange(0, max_hours + bin_step, bin_step)

# 绘制直方图（标准化为密度）
hist, bins, _ = plt.hist(
    valid_data,
    bins=bins,
    density=True,
    alpha=0.6,
    color='#1f77b4',
    edgecolor='white',
    label='频次分布'
)

# 计算并绘制KDE曲线
kde = gaussian_kde(valid_data)
x_range = np.linspace(0, max_hours, 1000)
plt.plot(x_range, kde(x_range), 
         color='#d62728', 
         linewidth=2.5,
         label='密度曲线')

# 添加关键统计指标
median_hours = np.median(valid_data)
mean_hours = np.mean(valid_data)

plt.axvline(median_hours, color='#2ca02c', linestyle='--', 
          linewidth=2, label=f'中位数: {median_hours:.1f}小时')
plt.axvline(mean_hours, color='#ff7f0e', linestyle='-.',
          linewidth=2, label=f'平均值: {mean_hours:.1f}小时')

# 添加分布特征注释
plt.text(x=0.95*max_hours, y=0.8*plt.ylim()[1],
        s=f'''数据特征：
        观测区间: 0-{max_hours}小时
        有效样本: {len(valid_data):,}条
        极端值阈值: >{max_hours}小时''',
        ha='right', va='top',
        bbox=dict(boxstyle='round', alpha=0.2, facecolor='white'))

# 图表美化
plt.title('优惠券使用时间间隔分布（小时粒度）\n[有效使用数据]', fontsize=15, pad=20)
plt.xlabel('领券到使用的时间间隔（小时）', fontsize=12)
plt.ylabel('概率密度', fontsize=12)
plt.legend(frameon=True, facecolor='#f0f0f0', loc='upper right')
plt.xticks(np.arange(0, max_hours+1, bin_step*4 if bin_step>1 else 24))  # 自动刻度
plt.grid(axis='y', alpha=0.3, linestyle='--')

# 显示图表
plt.tight_layout()
plt.show()

In [ ]:
merged_data.head()

In [ ]:
merged_data.columns

In [ ]:
merged_data.to_csv('processed_data_with_features.csv', index=False)
print("特征生成完成，并保存到'processed_data_with_features.csv'文件中。")

In [ ]:
from sklearn.ensemble import IsolationForest

# 加载预处理后的数据
data = pd.read_csv('processed_data_with_features.csv')

# 选择与薅羊毛行为相关的特征
features = ['Coupon_usage_rate', 'low_price_high_subsidy', 'coupon_usage_interval',
            'login_order_ratio', 'unused_coupon_count', 'coupon_acquisition_rate', 'Actual_pay','Price_limit','rule2_free_order','rule3_dormant','rule4_rapid_usage']

# 提取特征数据
X = data[features]

In [ ]:
data.head()

In [ ]:
# 查看每列的空值数量
missing_values = X.isna().sum()
missing_values

In [ ]:
X['login_order_ratio'].fillna(0, inplace=True)

In [ ]:
X['Price_limit'].fillna(0, inplace=True)

In [ ]:
# 使用Isolation Forest进行异常检测
iso_forest = IsolationForest(contamination=0.01, random_state=42)

In [ ]:
data['anomaly'] = iso_forest.fit_predict(X)

In [ ]:
# 查看异常用户
anomalies = data[data['anomaly'] == -1]
print(anomalies.head())

data.to_csv('divide_data.csv', index=False)

# 筛选出异常行为次数在十次以上的用户
anomaly_counts = data.groupby('User_id')['anomaly'].apply(lambda x: (x == -1).sum()).reset_index(name='anomaly_count')
users_to_keep = anomaly_counts[anomaly_counts['anomaly_count'] >= 20]['User_id']

# 保留这些用户的数据
filtered_data = data[data['User_id'].isin(users_to_keep)]

# 保存处理后的数据
filtered_data.to_csv('filtered_data.csv', index=False)

In [ ]:
filtered_data

# 不同类型异常画像

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from matplotlib.gridspec import GridSpec

# ==================== 数据加载与过滤 ====================
def load_and_filter_data(path):
    """加载数据并筛选异常用户"""
    df = pd.read_csv(path)
    
    # 统计用户异常次数
    anomaly_users = df[df['anomaly'] == -1].groupby('User_id').size()
    high_risk_users = anomaly_users[anomaly_users >=8].index
    
    # 筛选异常用户的所有记录
    filtered_df = df[df['User_id'].isin(high_risk_users)].copy()
    print(f"识别到高危用户数: {len(high_risk_users)}, 相关记录数: {len(filtered_df)}")
    return filtered_df

# ==================== 特征工程 ====================
def prepare_features(df):
    """数据预处理与特征工程"""
    features = [
        'Coupon_usage_rate', 'low_price_high_subsidy', 'coupon_usage_interval',
        'login_order_ratio', 'unused_coupon_count', 'coupon_acquisition_rate',
        'Actual_pay', 'Price_limit', 'rule2_free_order', 'rule3_dormant', 'rule4_rapid_usage'
    ]
    
    # 数据清洗
    df[features] = df[features].replace([np.inf, -np.inf], np.nan)
    df[features] = df[features].fillna(df[features].median())
    
    # 添加用户级聚合特征
    user_agg = df.groupby('User_id')[features].agg(['mean', 'max'])
    user_agg.columns = [f"{col}_{stat}" for col, stat in user_agg.columns]
    
    return pd.merge(df, user_agg, on='User_id'), features + list(user_agg.columns)

# ==================== 聚类分析 ====================
def cluster_analysis(X, max_clusters=8):
    """交互式聚类分析"""
    # 标准化处理
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)
    
    # 计算评估指标
    cluster_range = list(range(2, max_clusters+1))
    wcss, sil_scores = [], []
    
    for k in cluster_range:
        kmeans = KMeans(n_clusters=k, random_state=42).fit(X_scaled)
        wcss.append(kmeans.inertia_)
        sil_scores.append(silhouette_score(X_scaled, kmeans.labels_))
    
    # 可视化评估曲线
    plt.figure(figsize=(12,6))
    plt.plot(cluster_range, wcss, 'bo-', linewidth=2, markersize=8, label='WCSS')
    plt.xlabel('聚类数量', fontsize=12)
    plt.ylabel('WCSS', color='b', fontsize=12)
    plt.xticks(cluster_range)
    
    ax2 = plt.twinx()
    ax2.plot(cluster_range, sil_scores, 'r--', linewidth=2, markersize=8, label='轮廓系数')
    ax2.set_ylabel('轮廓系数', color='r', fontsize=12)
    plt.title('聚类评估曲线', fontsize=14)
    plt.show()
    
    # 交互选择聚类数
    n_clusters = int(input("请输入选择的聚类数: "))
    kmeans = KMeans(n_clusters=n_clusters, random_state=42).fit(X_scaled)
    return kmeans.labels_

# ==================== 可视化模块 ====================
def plot_feature_distribution(df, features):
    """特征分布箱线图"""
    plt.figure(figsize=(18, 12))
    colors = plt.cm.tab10(np.linspace(0, 1, df['cluster'].nunique()))
    
    for i, feat in enumerate(features[:6]):
        plt.subplot(2, 3, i+1)
        
        # 准备箱线图数据
        data = [df[df['cluster'] == c][feat] for c in sorted(df['cluster'].unique())]
        
        # 绘制箱线图
        bp = plt.boxplot(data, patch_artist=True, showfliers=False)
        
        # 设置样式
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
        plt.xticks(range(1, len(colors)+1), [f'Group {c}' for c in sorted(df['cluster'].unique())])
        plt.title(feat, fontsize=12)
    
    plt.tight_layout()
    plt.suptitle("特征分布对比", y=1.02, fontsize=16)
    plt.show()

def plot_feature_matrix(df, features):
    """特征关系矩阵图"""
    fig, axes = plt.subplots(5, 5, figsize=(20, 20))
    plt.subplots_adjust(hspace=0.4, wspace=0.4)
    features = features[:5]  # 展示前5个特征
    colors = plt.cm.tab10(np.linspace(0, 1, df['cluster'].nunique()))
    
    for i in range(5):
        for j in range(5):
            ax = axes[i, j]
            if i == j:
                # 对角线显示直方图
                ax.hist(df[features[i]], bins=30, color='skyblue', alpha=0.7)
                ax.set_title(features[i], fontsize=10)
            else:
                # 非对角线显示散点图
                for c, color in zip(sorted(df['cluster'].unique()), colors):
                    cluster_data = df[df['cluster'] == c]
                    ax.scatter(cluster_data[features[j]], cluster_data[features[i]], 
                              s=10, color=color, alpha=0.5, label=f'Group {c}')
                ax.set_xlabel(features[j], fontsize=8)
                ax.set_ylabel(features[i], fontsize=8)
                ax.tick_params(axis='both', labelsize=6)
    
    plt.suptitle("特征关系矩阵", y=0.93, fontsize=18)
    plt.show()

# ==================== 画像生成 ====================
def generate_profile(df, features):
    """生成详细特征画像报告"""
    # 定义统计函数
    stats_functions = {
        'count': ('数量', 'count'),
        'mean': ('均值', lambda x: f"{x.mean():.2f}"),
        'median': ('中位数', lambda x: f"{x.median():.2f}"),
        'std': ('标准差', lambda x: f"{x.std():.2f}"),
        'q90': ('90分位', lambda x: f"{x.quantile(0.9):.2f}")
    }
    
    # 特征中文映射
    feature_names = {
        'Coupon_usage_rate': '优惠券使用率(%)',
        'low_price_high_subsidy': '低价高补贴订单占比(%)',
        'coupon_usage_interval': '用券间隔(小时)',
        'login_order_ratio': '登录下单比',
        'unused_coupon_count': '未使用券数量',
        'coupon_acquisition_rate': '日均获券数',
        'Actual_pay': '实际支付金额(元)',
        'Price_limit': '价格限制(元)',
        'rule2_free_order': '零元订单比例(%)',
        'rule3_dormant': '静默天数占比(%)',
        'rule4_rapid_usage': '快速用券比例(%)'
    }
    
    # 生成统计报表
    report = {}
    for cluster in sorted(df['cluster'].unique()):
        cluster_data = df[df['cluster'] == cluster]
        cluster_report = []
        
        # 基本统计
        cluster_report.append(f"■ 群体 {cluster} 概况")
        cluster_report.append(f"- 用户数量: {len(cluster_data['User_id'].unique()):,}人")
        cluster_report.append(f"- 记录数量: {len(cluster_data):,}条")
        cluster_report.append("")
        
        # 详细特征分析
        cluster_report.append("■ 特征分析")
        for feature in features:
            if feature not in feature_names:
                continue
                
            values = cluster_data[feature]
            stats = []
            for stat_name, (stat_label, func) in stats_functions.items():
                if isinstance(func, str):
                    stat_value = getattr(values, func)()
                else:
                    try:
                        stat_value = func(values)
                    except:
                        stat_value = "N/A"
                stats.append(f"{stat_label}: {stat_value}")
            
            # 二进制特征特殊处理
            if feature in ['rule2_free_order', 'rule3_dormant', 'rule4_rapid_usage']:
                percentage = values.mean() * 100
                cluster_report.append(f"  ◆ {feature_names[feature]}")
                cluster_report.append(f"    ▷ 触发比例: {percentage:.1f}%")
            else:
                cluster_report.append(f"  ◆ {feature_names[feature]}")
                cluster_report.append("    ▷ " + " | ".join(stats))
            
            cluster_report.append("")
        
        report[cluster] = "\n".join(cluster_report)
    
    # 打印报告
    print("="*80)
    print("异常用户群体详细画像报告".center(70))
    print("="*80)
    for cluster in sorted(df['cluster'].unique()):
        print(f"\n{report[cluster]}")
        print("-"*80)
    print("="*80)

# ==================== 主流程 ====================
if __name__ == "__main__":
    # 数据加载与处理
    df = load_and_filter_data("divide_data.csv")
    df, features = prepare_features(df)
    
    # 聚类分析
    df['cluster'] = cluster_analysis(df[features])
    
    # 可视化分析
    plot_feature_distribution(df, features)
    plot_feature_matrix(df, features)
    
    # 生成报告
    generate_profile(df, features)
    df.to_csv("classified_anomalies.csv", index=False)
    print("分析结果已保存至 classified_anomalies.csv")

In [ ]:
df['cluster'].value_counts()

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi

# ==================== 数据加载 ====================
def load_clustered_data(path):
    """加载聚类结果数据"""
    df = pd.read_csv(path)
    return df

# ==================== 可视化模块 ====================
def plot_bar_chart(df, features):
    """绘制不同群体的特征均值条形图"""
    plt.figure(figsize=(18, 12))
    
    for i, feat in enumerate(features[:6]):
        plt.subplot(2, 3, i+1)
        
        # 计算每个群体的均值
        mean_values = df.groupby('cluster')[feat].mean()
        
        # 绘制条形图
        mean_values.plot(kind='bar', color=plt.cm.tab10(np.linspace(0, 1, df['cluster'].nunique())))
        plt.title(f'{feat} 均值比较', fontsize=12)
        plt.xlabel('群体', fontsize=10)
        plt.ylabel('均值', fontsize=10)
    
    plt.tight_layout()
    plt.suptitle("不同群体特征均值比较", y=1.02, fontsize=16)
    plt.savefig("特征均值条形图.jpg", dpi=300, bbox_inches='tight')  # 保存图片
    plt.show()

def plot_heatmap(df, features):
    """绘制不同群体的特征相关性热力图"""
    plt.figure(figsize=(15, 10))
    
    # 计算相关性矩阵
    corr_matrix = df[features + ['cluster']].corr()
    
    # 绘制热力图
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
    plt.title("特征相关性热力图", fontsize=14)
    plt.savefig("特征相关性热力图.jpg", dpi=300, bbox_inches='tight')  # 保存图片
    plt.show()



# ==================== 主流程 ====================
if __name__ == "__main__":
    # 数据加载
    df = load_clustered_data("classified_anomalies.csv")
    
    # 特征列表
    features = [
        'Coupon_usage_rate', 'low_price_high_subsidy', 'coupon_usage_interval',
        'login_order_ratio', 'unused_coupon_count', 'coupon_acquisition_rate',
        'Actual_pay', 'Price_limit', 'rule2_free_order', 'rule3_dormant', 'rule4_rapid_usage'
    ]
    
    # 可视化分析
    plot_bar_chart(df, features)
    plot_heatmap(df, features)

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ==================== 数据加载 ====================
def load_and_filter_data(path):
    """加载数据并筛选正常用户和异常用户"""
    df = pd.read_csv(path)
    
    # 统计用户异常次数
    anomaly_counts = df[df['anomaly'] == -1].groupby('User_id').size()
    
    # 筛选正常用户（异常次数小于8）
    normal_users = anomaly_counts[anomaly_counts < 8].index
    normal_df = df[df['User_id'].isin(normal_users)].copy()
    
    # 筛选异常用户（异常次数大于等于8）
    high_risk_users = anomaly_counts[anomaly_counts >= 8].index
    high_risk_df = df[df['User_id'].isin(high_risk_users)].copy()
    
    return normal_df, high_risk_df

def load_clustered_data(path):
    """加载聚类结果数据"""
    df = pd.read_csv(path)
    return df

# ==================== 可视化模块 ====================
def plot_bar_chart(normal_df, clustered_df, features):
    """绘制正常用户和各类异常用户的特征均值条形图"""
    plt.figure(figsize=(18, 12))
    
    # 计算正常用户的均值
    normal_means = normal_df[features].mean()
    
    # 按聚类分组计算异常用户的均值
    cluster_means = clustered_df.groupby('cluster')[features].mean()
    
    # 绘制每个特征的条形图
    for i, feat in enumerate(features[:6]):
        plt.subplot(2, 3, i+1)
        
        # 准备数据
        data = [normal_means[feat]] + cluster_means[feat].tolist()
        labels = ['正常用户'] + [f'异常群 {c}' for c in sorted(clustered_df['cluster'].unique())]
        
        # 绘制条形图
        plt.bar(labels, data, color=plt.cm.tab10(np.linspace(0, 1, len(labels))))
        plt.title(f'{feat} 均值比较', fontsize=12)
        plt.xlabel('用户类型', fontsize=10)
        plt.ylabel('均值', fontsize=10)
    
    plt.tight_layout()
    plt.suptitle("正常用户与各类异常用户特征均值比较", y=1.02, fontsize=16)
    plt.savefig("特征均值条形图.jpg", dpi=300, bbox_inches='tight')  # 保存图片
    plt.show()

def plot_heatmap(normal_df, clustered_df, features):
    """绘制正常用户和各类异常用户的特征相关性热力图"""
    plt.figure(figsize=(15, 10))
    
    # 计算正常用户的相关性矩阵
    plt.subplot(1, 2, 1)
    corr_normal = normal_df[features].corr()
    sns.heatmap(corr_normal, annot=True, cmap='coolwarm', linewidths=0.5)
    plt.title("正常用户特征相关性热力图", fontsize=12)
    
    # 计算异常用户的相关性矩阵
    plt.subplot(1, 2, 2)
    corr_high_risk = clustered_df[features].corr()
    sns.heatmap(corr_high_risk, annot=True, cmap='coolwarm', linewidths=0.5)
    plt.title("异常用户特征相关性热力图", fontsize=12)
    
    plt.tight_layout()
    plt.suptitle("特征相关性热力图对比", y=1.05, fontsize=16)
    plt.savefig("特征相关性热力图对比.jpg", dpi=300, bbox_inches='tight')  # 保存图片
    plt.show()

# ==================== 主流程 ====================
if __name__ == "__main__":
    # 数据加载
    normal_df, _ = load_and_filter_data("divide_data.csv")
    clustered_df = load_clustered_data("classified_anomalies.csv")
    
    # 特征列表
    features = [
        'Coupon_usage_rate', 'low_price_high_subsidy', 'coupon_usage_interval',
        'login_order_ratio', 'unused_coupon_count', 'coupon_acquisition_rate',
        'Actual_pay', 'Price_limit', 'rule2_free_order', 'rule3_dormant', 'rule4_rapid_usage'
    ]
    
    # 输出用户数量和记录条数
    print(f"正常用户数量: {len(normal_df['User_id'].unique())}, 记录条数: {len(normal_df)}")
    
    # 按聚类分组输出异常用户数量和记录条数
    for cluster in sorted(clustered_df['cluster'].unique()):
        cluster_data = clustered_df[clustered_df['cluster'] == cluster]
        print(f"异常群 {cluster} 用户数量: {len(cluster_data['User_id'].unique())}, 记录条数: {len(cluster_data)}")
    
    # 可视化分析
    plot_bar_chart(normal_df, clustered_df, features)
    plot_heatmap(normal_df, clustered_df, features)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# ==================== 数据加载 ====================
def load_and_filter_data(path):
    """加载数据并筛选正常用户和异常用户"""
    df = pd.read_csv(path)
    
    # 统计用户异常次数
    anomaly_counts = df[df['anomaly'] == -1].groupby('User_id').size()
    
    # 筛选正常用户（异常次数小于8）
    normal_users = anomaly_counts[anomaly_counts < 8].index
    normal_df = df[df['User_id'].isin(normal_users)].copy()
    
    return normal_df

def load_clustered_data(path):
    """加载聚类结果数据"""
    df = pd.read_csv(path)
    return df

# ==================== 计算平均值 ====================
def calculate_averages(normal_df, clustered_df):
    """计算正常用户和各类异常用户的特征平均值"""
    # 特征列表
    features = [
        'Coupon_usage_rate', 'low_price_high_subsidy', 'coupon_usage_interval',
        'login_order_ratio', 'unused_coupon_count', 'coupon_acquisition_rate',
        'Actual_pay', 'Price_limit', 'rule2_free_order', 'rule3_dormant', 'rule4_rapid_usage'
    ]
    
    # 正常用户平均值
    normal_averages = normal_df[features].mean()
    
    # 异常用户按群体分组计算平均值
    cluster_averages = {}
    for cluster in sorted(clustered_df['cluster'].unique()):
        cluster_data = clustered_df[clustered_df['cluster'] == cluster]
        cluster_averages[cluster] = cluster_data[features].mean()
    
    return normal_averages, cluster_averages

# ==================== 绘制雷达图 ====================
def plot_radar_chart(normal_averages, cluster_averages):
    """绘制雷达图"""
    # 特征列表
    features = [
        'Coupon_usage_rate', 'low_price_high_subsidy', 'coupon_usage_interval',
        'login_order_ratio', 'unused_coupon_count', 'coupon_acquisition_rate',
        'Actual_pay', 'Price_limit', 'rule2_free_order', 'rule3_dormant', 'rule4_rapid_usage'
    ]
    
    # 特征中文映射
    feature_names = {
        'Coupon_usage_rate': '优惠券使用率',
        'low_price_high_subsidy': '低价高补贴订单占比',
        'coupon_usage_interval': '用券间隔',
        'login_order_ratio': '登录下单比',
        'unused_coupon_count': '未使用券数量',
        'coupon_acquisition_rate': '日均获券数',
        'Actual_pay': '实际支付金额',
        'Price_limit': '价格限制',
        'rule2_free_order': '零元订单比例',
        'rule3_dormant': '静默天数占比',
        'rule4_rapid_usage': '快速用券比例'
    }
    
    # 将特征名称转换为中文
    labels = [feature_names[feature] for feature in features]
    
    # 正常用户平均值
    normal_values = normal_averages.values
    
    # 异常用户群体的平均值
    cluster_values = [cluster_averages[cluster].values for cluster in cluster_averages.keys()]
    
    # 合并数据进行标准化
    data = np.array([normal_values] + cluster_values)
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)
    
    # 提取标准化后的数据
    normal_scaled = data_scaled[0]
    cluster_scaled = data_scaled[1:]
    
    # 绘制雷达图
    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    
    # 闭合雷达图
    normal_scaled = np.concatenate((normal_scaled, normal_scaled[:1]))
    closed_cluster_scaled = []
    for values in cluster_scaled:
        closed_values = np.concatenate((values, values[:1]))
        closed_cluster_scaled.append(closed_values)
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(polar=True))
    
    # 绘制正常用户
    ax.fill(angles, normal_scaled, color='skyblue', alpha=0.3, label='正常用户')
    ax.plot(angles, normal_scaled, color='skyblue', linewidth=2)
    
    # 绘制异常用户群体
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
    for i, cluster in enumerate(cluster_averages.keys()):
        if i < len(closed_cluster_scaled):
            ax.fill(angles, closed_cluster_scaled[i], color=colors[i], alpha=0.25, label=f'异常用户群体 {cluster}')
            ax.plot(angles, closed_cluster_scaled[i], color=colors[i], linewidth=2)
    
    # 设置特征标签
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=12, fontfamily='sans-serif')
    
    # 设置雷达图的范围
    max_val = max([max(c) for c in closed_cluster_scaled] + [max(normal_scaled)]) + 0.1
    min_val = min([min(c) for c in closed_cluster_scaled] + [min(normal_scaled)]) - 0.1
    ax.set_ylim(min_val, max_val)
    
    # 添加网格线
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.5)
    
    # 设置背景颜色
    ax.set_facecolor('#f8f8f8')
    
    # 设置图例
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=12, frameon=True, shadow=True)
    
    # 添加标题
    plt.title('正常用户与异常用户群体的雷达图（Z-score标准化）', fontsize=18, pad=20)
    
    # 保存图片
    plt.savefig("雷达图优化版.jpg", dpi=300, bbox_inches='tight')
    
    plt.tight_layout()
    plt.show()

# ==================== 输出平均值 ====================
def print_averages(normal_averages, cluster_averages):
    """输出正常用户和各类异常用户的平均值"""
    # 特征列表
    features = [
        'Coupon_usage_rate', 'low_price_high_subsidy', 'coupon_usage_interval',
        'login_order_ratio', 'unused_coupon_count', 'coupon_acquisition_rate',
        'Actual_pay', 'Price_limit', 'rule2_free_order', 'rule3_dormant', 'rule4_rapid_usage'
    ]
    
    # 特征中文映射
    feature_names = {
        'Coupon_usage_rate': '优惠券使用率',
        'low_price_high_subsidy': '低价高补贴订单占比',
        'coupon_usage_interval': '用券间隔',
        'login_order_ratio': '登录下单比',
        'unused_coupon_count': '未使用券数量',
        'coupon_acquisition_rate': '日均获券数',
        'Actual_pay': '实际支付金额',
        'Price_limit': '价格限制',
        'rule2_free_order': '零元订单比例',
        'rule3_dormant': '静默天数占比',
        'rule4_rapid_usage': '快速用券比例'
    }
    
    # 输出正常用户平均值
    print("正常用户平均值:")
    for feature in features:
        print(f"{feature_names[feature]}: {normal_averages[feature]:.4f}")
    print("\n")
    
    # 输出异常用户群体平均值
    for cluster in cluster_averages.keys():
        print(f"异常用户群体 {cluster} 平均值:")
        for feature in features:
            print(f"{feature_names[feature]}: {cluster_averages[cluster][feature]:.4f}")
        print("\n")

# ==================== 主流程 ====================
if __name__ == "__main__":
    # 数据加载
    normal_df = load_and_filter_data("divide_data.csv")
    clustered_df = load_clustered_data("classified_anomalies.csv")
    
    # 计算平均值
    normal_averages, cluster_averages = calculate_averages(normal_df, clustered_df)
    
    # 输出平均值
    print_averages(normal_averages, cluster_averages)
    
    # 绘制雷达图
    plot_radar_chart(normal_averages, cluster_averages)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# ==================== 数据加载 ====================
def load_and_filter_data(path):
    """加载数据并筛选正常用户和异常用户"""
    df = pd.read_csv(path)
    
    # 统计用户异常次数
    anomaly_counts = df[df['anomaly'] == -1].groupby('User_id').size()
    
    # 筛选正常用户（异常次数小于8）
    normal_users = anomaly_counts[anomaly_counts < 8].index
    normal_df = df[df['User_id'].isin(normal_users)].copy()
    
    return normal_df

def load_clustered_data(path):
    """加载聚类结果数据"""
    df = pd.read_csv(path)
    return df

# ==================== 计算平均值 ====================
def calculate_averages(normal_df, clustered_df):
    """计算正常用户和各类异常用户的特征平均值"""
    # 特征列表
    features = [
        'Coupon_usage_rate', 'low_price_high_subsidy', 'coupon_usage_interval',
        'login_order_ratio', 'unused_coupon_count', 'coupon_acquisition_rate',
        'Actual_pay', 'Price_limit', 'rule2_free_order', 'rule3_dormant', 'rule4_rapid_usage'
    ]
    
    # 正常用户平均值
    normal_averages = normal_df[features].mean()
    
    # 异常用户按群体分组计算平均值
    cluster_averages = {}
    for cluster in sorted(clustered_df['cluster'].unique()):
        cluster_data = clustered_df[clustered_df['cluster'] == cluster]
        cluster_averages[cluster] = cluster_data[features].mean()
    
    return normal_averages, cluster_averages

# ==================== 绘制雷达图 ====================
def plot_radar_chart(normal_averages, cluster_averages):
    """绘制雷达图"""
    # 特征列表
    features = [
        'Coupon_usage_rate', 'low_price_high_subsidy', 'coupon_usage_interval',
        'login_order_ratio', 'unused_coupon_count', 'coupon_acquisition_rate',
        'Actual_pay', 'Price_limit', 'rule2_free_order', 'rule3_dormant', 'rule4_rapid_usage'
    ]
    
    # 特征中文映射
    feature_names = {
        'Coupon_usage_rate': '优惠券使用率',
        'low_price_high_subsidy': '低价高补贴订单占比',
        'coupon_usage_interval': '用券间隔',
        'login_order_ratio': '登录下单比',
        'unused_coupon_count': '未使用券数量',
        'coupon_acquisition_rate': '日均获券数',
        'Actual_pay': '实际支付金额',
        'Price_limit': '价格限制',
        'rule2_free_order': '零元订单比例',
        'rule3_dormant': '静默天数占比',
        'rule4_rapid_usage': '快速用券比例'
    }
    
    # 将特征名称转换为中文
    labels = [feature_names[feature] for feature in features]
    
    # 正常用户平均值
    normal_values = normal_averages.values
    
    # 异常用户群体的平均值
    cluster_values = [cluster_averages[cluster].values for cluster in cluster_averages.keys()]
    
    # 合并数据进行标准化
    data = np.array([normal_values] + cluster_values)
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)
    
    # 提取标准化后的数据
    normal_scaled = data_scaled[0]
    cluster_scaled = data_scaled[1:]
    
    # 绘制雷达图
    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    
    # 闭合雷达图
    normal_scaled = np.concatenate((normal_scaled, normal_scaled[:1]))
    closed_cluster_scaled = []
    for values in cluster_scaled:
        closed_values = np.concatenate((values, values[:1]))
        closed_cluster_scaled.append(closed_values)
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(polar=True))
    
    # 绘制正常用户
    ax.fill(angles, normal_scaled, color='skyblue', alpha=0.3, label='正常用户')
    ax.plot(angles, normal_scaled, color='skyblue', linewidth=2)
    
    # 绘制异常用户群体
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
    for i, cluster in enumerate(cluster_averages.keys()):
        if i < len(closed_cluster_scaled):
            ax.fill(angles, closed_cluster_scaled[i], color=colors[i], alpha=0.25, label=f'异常用户群体 {cluster}')
            ax.plot(angles, closed_cluster_scaled[i], color=colors[i], linewidth=2)
    
    # 设置特征标签
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=12, fontfamily='sans-serif')
    
    # 设置雷达图的范围
    max_val = max([max(c) for c in closed_cluster_scaled] + [max(normal_scaled)]) + 0.1
    min_val = min([min(c) for c in closed_cluster_scaled] + [min(normal_scaled)]) - 0.1
    ax.set_ylim(min_val, max_val)
    
    # 添加网格线
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.5)
    
    # 设置背景颜色
    ax.set_facecolor('#f8f8f8')
    
    # 设置图例
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=12, frameon=True, shadow=True)
    
    # 添加标题
    plt.title('正常用户与异常用户群体的雷达图（Z-score标准化）', fontsize=18, pad=20)
    
    # 保存图片
    plt.savefig("雷达图优化版.jpg", dpi=300, bbox_inches='tight')
    
    plt.tight_layout()
    plt.show()

# ==================== 主流程 ====================
if __name__ == "__main__":
    # 数据加载
    normal_df = load_and_filter_data("divide_data.csv")
    clustered_df = load_clustered_data("classified_anomalies.csv")
    
    # 计算平均值
    normal_averages, cluster_averages = calculate_averages(normal_df, clustered_df)
    
    # 绘制雷达图
    plot_radar_chart(normal_averages, cluster_averages)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# ==================== 数据加载 ====================
def load_and_filter_data(path):
    """加载数据并筛选正常用户和异常用户"""
    df = pd.read_csv(path)
    
    # 统计用户异常次数
    anomaly_counts = df[df['anomaly'] == -1].groupby('User_id').size()
    
    # 筛选正常用户（异常次数小于8）
    normal_users = anomaly_counts[anomaly_counts < 8].index
    normal_df = df[df['User_id'].isin(normal_users)].copy()
    
    return normal_df

def load_clustered_data(path):
    """加载聚类结果数据"""
    df = pd.read_csv(path)
    return df

# ==================== 计算平均值 ====================
def calculate_averages(normal_df, clustered_df):
    """计算正常用户和各类异常用户的特征平均值"""
    # 特征列表
    features = [
        'Coupon_usage_rate', 'low_price_high_subsidy', 'coupon_usage_interval',
        'login_order_ratio', 'unused_coupon_count', 'coupon_acquisition_rate',
        'Actual_pay', 'Price_limit', 'rule2_free_order', 'rule3_dormant', 'rule4_rapid_usage'
    ]
    
    # 正常用户平均值
    normal_averages = normal_df[features].mean()
    
    # 异常用户按群体分组计算平均值
    cluster_averages = {}
    for cluster in sorted(clustered_df['cluster'].unique()):
        cluster_data = clustered_df[clustered_df['cluster'] == cluster]
        cluster_averages[cluster] = cluster_data[features].mean()
    
    return normal_averages, cluster_averages

# ==================== 绘制雷达图 ====================
def plot_radar_chart(normal_averages, cluster_averages, normal_count, cluster_counts):
    """绘制雷达图"""
    # 特征列表
    features = [
        'Coupon_usage_rate', 'low_price_high_subsidy', 'coupon_usage_interval',
        'login_order_ratio', 'unused_coupon_count', 'coupon_acquisition_rate',
        'Actual_pay', 'Price_limit', 'rule2_free_order', 'rule3_dormant', 'rule4_rapid_usage'
    ]
    
    # 特征中文映射
    feature_names = {
        'Coupon_usage_rate': '优惠券使用率',
        'low_price_high_subsidy': '低价高补贴订单占比',
        'coupon_usage_interval': '用券间隔',
        'login_order_ratio': '登录下单比',
        'unused_coupon_count': '未使用券数量',
        'coupon_acquisition_rate': '日均获券数',
        'Actual_pay': '实际支付金额',
        'Price_limit': '价格限制',
        'rule2_free_order': '零元订单比例',
        'rule3_dormant': '静默天数占比',
        'rule4_rapid_usage': '快速用券比例'
    }
    
    # 将特征名称转换为中文
    labels = [feature_names[feature] for feature in features]
    
    # 正常用户平均值
    normal_values = normal_averages.values
    
    # 异常用户群体的平均值
    cluster_values = [cluster_averages[cluster].values for cluster in cluster_averages.keys()]
    
    # 合并数据进行标准化
    data = np.array([normal_values] + cluster_values)
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)
    
    # 提取标准化后的数据
    normal_scaled = data_scaled[0]
    cluster_scaled = data_scaled[1:]
    
    # 绘制雷达图
    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    
    # 闭合雷达图
    normal_scaled = np.concatenate((normal_scaled, normal_scaled[:1]))
    closed_cluster_scaled = []
    for values in cluster_scaled:
        closed_values = np.concatenate((values, values[:1]))
        closed_cluster_scaled.append(closed_values)
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(polar=True))
    
    # 绘制正常用户（紫色）
    ax.fill(angles, normal_scaled, color='purple', alpha=0.3, label=f'正常用户 ({normal_count}人)')
    ax.plot(angles, normal_scaled, color='purple', linewidth=2, marker='o', markersize=5)  # 添加点
    
    # 绘制异常用户群体
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
    for i, cluster in enumerate(cluster_averages.keys()):
        if i < len(closed_cluster_scaled):
            ax.fill(angles, closed_cluster_scaled[i], color=colors[i], alpha=0.25, label=f'异常用户群体 {cluster} ({cluster_counts[cluster]}人)')
            ax.plot(angles, closed_cluster_scaled[i], color=colors[i], linewidth=2, marker='o', markersize=5)  # 添加点
    
    # 设置特征标签
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=12, fontfamily='sans-serif')
    
    # 设置雷达图的范围
    max_val = max([max(c) for c in closed_cluster_scaled] + [max(normal_scaled)]) + 0.1
    min_val = min([min(c) for c in closed_cluster_scaled] + [min(normal_scaled)]) - 0.1
    ax.set_ylim(min_val, max_val)
    
    # 添加网格线
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.5)
    
    # 设置背景颜色
    ax.set_facecolor('#f8f8f8')
    
    # 设置图例
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=12, frameon=True, shadow=True)
    
    # 添加标题
    plt.title('正常用户与异常用户群体的雷达图（Z-score标准化）', fontsize=18, pad=20)
    
    # 保存图片
    plt.savefig("雷达图优化版.jpg", dpi=300, bbox_inches='tight')
    
    plt.tight_layout()
    plt.show()

# ==================== 输出平均值 ====================
def print_averages(normal_averages, cluster_averages, normal_count, cluster_counts):
    """输出正常用户和各类异常用户的平均值"""
    # 特征列表
    features = [
        'Coupon_usage_rate', 'low_price_high_subsidy', 'coupon_usage_interval',
        'login_order_ratio', 'unused_coupon_count', 'coupon_acquisition_rate',
        'Actual_pay', 'Price_limit', 'rule2_free_order', 'rule3_dormant', 'rule4_rapid_usage'
    ]
    
    # 特征中文映射
    feature_names = {
        'Coupon_usage_rate': '优惠券使用率',
        'low_price_high_subsidy': '低价高补贴订单占比',
        'coupon_usage_interval': '用券间隔',
        'login_order_ratio': '登录下单比',
        'unused_coupon_count': '未使用券数量',
        'coupon_acquisition_rate': '日均获券数',
        'Actual_pay': '实际支付金额',
        'Price_limit': '价格限制',
        'rule2_free_order': '零元订单比例',
        'rule3_dormant': '静默天数占比',
        'rule4_rapid_usage': '快速用券比例'
    }
    
    # 输出正常用户平均值
    print(f"正常用户 ({normal_count}人) 平均值:")
    for feature in features:
        print(f"{feature_names[feature]}: {normal_averages[feature]:.4f}")
    print("\n")
    
    # 输出异常用户群体平均值
    for cluster in cluster_averages.keys():
        print(f"异常用户群体 {cluster} ({cluster_counts[cluster]}人) 平均值:")
        for feature in features:
            print(f"{feature_names[feature]}: {cluster_averages[cluster][feature]:.4f}")
        print("\n")

# ==================== 主流程 ====================
if __name__ == "__main__":
    # 数据加载
    normal_df = load_and_filter_data("divide_data.csv")
    clustered_df = load_clustered_data("classified_anomalies.csv")
    
    # 计算用户数量
    normal_count = len(normal_df['User_id'].unique())
    cluster_counts = {}
    for cluster in sorted(clustered_df['cluster'].unique()):
        cluster_data = clustered_df[clustered_df['cluster'] == cluster]
        cluster_counts[cluster] = len(cluster_data['User_id'].unique())
    
    # 计算平均值
    normal_averages, cluster_averages = calculate_averages(normal_df, clustered_df)
    
    # 输出平均值
    print_averages(normal_averages, cluster_averages, normal_count, cluster_counts)
    
    # 绘制雷达图
    plot_radar_chart(normal_averages, cluster_averages, normal_count, cluster_counts)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# ==================== 数据加载 ====================
def load_and_filter_data(path):
    """加载数据并筛选正常用户和异常用户"""
    df = pd.read_csv(path)

    # 统计用户异常次数
    anomaly_counts = df[df['anomaly'] == -1].groupby('User_id').size()

    # 筛选正常用户（异常次数小于8）
    normal_users = anomaly_counts[anomaly_counts < 8].index
    normal_df = df[df['User_id'].isin(normal_users)].copy()

    return normal_df

def load_clustered_data(path):
    """加载聚类结果数据"""
    df = pd.read_csv(path)
    return df

# ==================== 计算平均值 ====================
def calculate_averages(normal_df, clustered_df):
    """计算正常用户和各类异常用户的特征平均值"""
    # 特征列表，删除登录订单比
    features = [
        'Coupon_usage_rate', 'low_price_high_subsidy', 'coupon_usage_interval',
        'unused_coupon_count', 'coupon_acquisition_rate',
        'Actual_pay', 'Price_limit', 'rule2_free_order', 'rule3_dormant', 'rule4_rapid_usage'
    ]

    # 正常用户平均值
    normal_averages = normal_df[features].mean()

    # 异常用户按群体分组计算平均值
    cluster_averages = {}
    for cluster in sorted(clustered_df['cluster'].unique()):
        cluster_data = clustered_df[clustered_df['cluster'] == cluster]
        cluster_averages[cluster] = cluster_data[features].mean()

    return normal_averages, cluster_averages

# ==================== 绘制雷达图 ====================
def plot_radar_chart(normal_averages, cluster_averages, normal_count, cluster_counts):
    """绘制雷达图"""
    # 特征列表，删除登录订单比
    features = [
        'Coupon_usage_rate', 'low_price_high_subsidy', 'coupon_usage_interval', 'unused_coupon_count', 'coupon_acquisition_rate',
        'Actual_pay', 'Price_limit', 'rule2_free_order', 'rule3_dormant', 'rule4_rapid_usage'
    ]

    # 特征中文映射
    feature_names = {
        'Coupon_usage_rate': '优惠券使用率',
        'low_price_high_subsidy': '低价高补贴订单占比',
        'coupon_usage_interval': '用券间隔',
        'unused_coupon_count': '未使用券数量',
        'coupon_acquisition_rate': '日均获券数',
        'Actual_pay': '实际支付金额',
        'Price_limit': '价格限制',
        'rule2_free_order': '零元订单比例',
        'rule3_dormant': '静默天数占比',
        'rule4_rapid_usage': '快速用券比例'
    }

    # 将特征名称转换为中文
    labels = [feature_names[feature] for feature in features]

    # 正常用户平均值
    normal_values = normal_averages.values

    # 异常用户群体的平均值
    cluster_values = [cluster_averages[cluster].values for cluster in cluster_averages.keys()]

    # 合并数据进行标准化
    data = np.array([normal_values] + cluster_values)
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)

    # 提取标准化后的数据
    normal_scaled = data_scaled[0]
    cluster_scaled = data_scaled[1:]

    # 绘制雷达图
    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()

    # 闭合雷达图
    normal_scaled = np.concatenate((normal_scaled, normal_scaled[:1]))
    closed_cluster_scaled = []
    for values in cluster_scaled:
        closed_values = np.concatenate((values, values[:1]))
        closed_cluster_scaled.append(closed_values)
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(polar=True))

    # 绘制正常用户（紫色）
    ax.fill(angles, normal_scaled, color='purple', alpha=0.3, label=f'正常用户 ({normal_count}人)')
    ax.plot(angles, normal_scaled, color='purple', linewidth=2, marker='o', markersize=5)  # 添加点

    # 绘制异常用户群体
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
    for i, cluster in enumerate(cluster_averages.keys()):
        if i < len(closed_cluster_scaled):
            ax.fill(angles, closed_cluster_scaled[i], color=colors[i], alpha=0.25, label=f'异常用户群体 {cluster} ({cluster_counts[cluster]}人)')
            ax.plot(angles, closed_cluster_scaled[i], color=colors[i], linewidth=2, marker='o', markersize=5)  # 添加点

    # 设置特征标签
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=12, fontfamily='sans-serif')

    # 设置雷达图的范围
    max_val = max([max(c) for c in closed_cluster_scaled] + [max(normal_scaled)]) + 0.1
    min_val = min([min(c) for c in closed_cluster_scaled] + [min(normal_scaled)]) - 0.1
    ax.set_ylim(min_val, max_val)

    # 添加网格线
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.5)

    # 设置背景颜色
    ax.set_facecolor('#f8f8f8')

    # 设置图例
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=12, frameon=True, shadow=True)

    # 添加标题
    plt.title('正常用户与异常用户群体的雷达图（Z - score标准化）', fontsize=18, pad=20)

    # 保存图片
    plt.savefig("雷达图优化版.jpg", dpi=300, bbox_inches='tight')

    plt.tight_layout()
    plt.show()

# ==================== 输出平均值 ====================
def print_averages(normal_averages, cluster_averages, normal_count, cluster_counts):
    """输出正常用户和各类异常用户的平均值"""
    # 特征列表，删除登录订单比
    features = [
        'Coupon_usage_rate', 'low_price_high_subsidy', 'coupon_usage_interval',
        'unused_coupon_count', 'coupon_acquisition_rate',
        'Actual_pay', 'Price_limit', 'rule2_free_order', 'rule3_dormant', 'rule4_rapid_usage'
    ]

    # 特征中文映射
    feature_names = {
        'Coupon_usage_rate': '优惠券使用率',
        'low_price_high_subsidy': '低价高补贴订单占比',
        'coupon_usage_interval': '用券间隔',
        'unused_coupon_count': '未使用券数量',
        'coupon_acquisition_rate': '日均获券数',
        'Actual_pay': '实际支付金额',
        'Price_limit': '价格限制',
        'rule2_free_order': '零元订单比例',
        'rule3_dormant': '静默天数占比',
        'rule4_rapid_usage': '快速用券比例'
    }

    # 输出正常用户平均值
    print(f"正常用户 ({normal_count}人) 平均值:")
    for feature in features:
        print(f"{feature_names[feature]}: {normal_averages[feature]:.4f}")
    print("\n")

    # 输出异常用户群体平均值
    for cluster in cluster_averages.keys():
        print(f"异常用户群体 {cluster} ({cluster_counts[cluster]}人) 平均值:")
        for feature in features:
            print(f"{feature_names[feature]}: {cluster_averages[cluster][feature]:.4f}")
        print("\n")

# ==================== 主流程 ====================
if __name__ == "__main__":
    # 数据加载
    normal_df = load_and_filter_data("divide_data.csv")
    clustered_df = load_clustered_data("classified_anomalies.csv")

    # 计算用户数量
    normal_count = len(normal_df['User_id'].unique())
    cluster_counts = {}
    for cluster in sorted(clustered_df['cluster'].unique()):
        cluster_data = clustered_df[clustered_df['cluster'] == cluster]
        cluster_counts[cluster] = len(cluster_data['User_id'].unique())

    # 计算平均值
    normal_averages, cluster_averages = calculate_averages(normal_df, clustered_df)

    # 输出平均值
    print_averages(normal_averages, cluster_averages, normal_count, cluster_counts)

    # 绘制雷达图
    plot_radar_chart(normal_averages, cluster_averages, normal_count, cluster_counts)
    

# 用户画像绘图

In [ ]:
from PIL import Image, ImageDraw, ImageFont

# 定义特征词汇列表
feature_words = ["有耐心", "善于学习", "沟通能力强", "富有创造力"]

# 创建一个空白图像，设置宽度、高度和背景颜色（这里使用白色背景，RGB值为(255, 255, 255)）
width, height = 400, 600
image = Image.new('RGB', (width, height), color=(255, 255, 255))
draw = ImageDraw.Draw(image)

# 定义人物半身像轮廓的顶点坐标（简单示例，可根据需求调整）
person_outline = [
    (100, 100),  # 头部左上角
    (300, 100),  # 头部右上角
    (200, 200),  # 头部底部中点
    (100, 500),  # 身体左侧底部
    (300, 500)  # 身体右侧底部
]
draw.polygon(person_outline, outline=(0, 0, 0), width=5)  # 绘制人物轮廓，黑色线条，宽度为5

# 设置字体和字号（这里使用系统默认字体，可根据实际情况替换）
font = ImageFont.load_default()

# 确定词汇在人像内的起始位置
x = 120
y = 120

# 在人像内绘制每个特征词汇
for word in feature_words:
    draw.text((x, y), word, fill=(0, 0, 0), font=font)
    y += 30  # 调整垂直位置，使词汇之间有一定间隔

# 保存图片
output_path = "user_portrait_with_features.png"
image.save(output_path)
print(f"图片已保存到 {output_path}")